# LungCT COPD02 pTVPrep Visual Audit

This notebook adapts the 4DCT pTVPrep visual-audit convention to COPD case 02. It focuses on whether the pTV-style preprocessing/crop could explain the unusually poor pTVReg and MR-D-BSCP case-02 TRE.

Display convention follows `lungct_4dct_ptvprep_visual_audit.ipynb`: coronal views use radiology orientation with superior at top, inferior at bottom, patient right on image left, and patient left on image right.

To re-execute this notebook, set `INRORSPLINE_REPO_ROOT`, `COPD02_PTVPREP_CASE_DIR`, and `COPD02_PTVREG_MAT` to the repo root, COPD02 pTVPrep case folder, and selected pTVReg rerun deformation `.mat`, respectively. This keeps private host paths out of the notebook source while preserving a reproducible review surface.

The cyan/yellow landmark markers show the pTVReg-best outlier landmarks, not all landmarks:

- cyan circle: fixed iBH landmark with pTVReg snap TRE > 10 mm;
- yellow x: corresponding moving eBH landmark for the same outlier.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, display
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D


def required_env(name: str) -> Path:
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(f'Set {name} before running this notebook.')
    return Path(value)


REPO = required_env('INRORSPLINE_REPO_ROOT')
CASE_DIR = required_env('COPD02_PTVPREP_CASE_DIR')
PTVREG_MAT = required_env('COPD02_PTVREG_MAT')
OUT_DIR = REPO / "experiments/2026-08-03_dual_inr_copd_six_arm/preprocessing_audit"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_PATH = OUT_DIR / "copd02_ptv_cropped_fixed_moving_coronal_audit.png"

display(Markdown('**COPD02 pTVPrep root:** `$COPD02_PTVPREP_CASE_DIR`'))
display(Markdown('**pTVReg rerun MAT:** `$COPD02_PTVREG_MAT`'))
display(Markdown('**Figure output:** `$INRORSPLINE_REPO_ROOT/experiments/2026-08-03_dual_inr_copd_six_arm/preprocessing_audit/copd02_ptv_cropped_fixed_moving_coronal_audit.png`'))


In [ ]:
fixed_crop = np.load(CASE_DIR / "fixed_iBH_crop_ptvintensity_zyx.npy")
moving_crop = np.load(CASE_DIR / "moving_eBH_crop_ptvintensity_zyx.npy")
fixed_reg = np.load(CASE_DIR / "fixed_iBH_reg1mm_trilinear_zyx.npy")
moving_reg = np.load(CASE_DIR / "moving_eBH_reg1mm_trilinear_zyx.npy")

fixed_mask_crop = np.load(CASE_DIR / "fixed_iBH_lungmask_crop_zyx.npy")
moving_mask_crop = np.load(CASE_DIR / "moving_eBH_lungmask_crop_zyx.npy")
fixed_mask_reg = np.load(CASE_DIR / "fixed_iBH_lungmask_reg1mm_nearest_zyx.npy")
moving_mask_reg = np.load(CASE_DIR / "moving_eBH_lungmask_reg1mm_nearest_zyx.npy")

lm_fix_crop = np.load(CASE_DIR / "landmarks_fixed_iBH_crop_zyx.npy")
lm_mov_crop = np.load(CASE_DIR / "landmarks_moving_eBH_crop_zyx.npy")
lm_fix_reg = np.load(CASE_DIR / "landmarks_fixed_iBH_reg1mm_zyx.npy")
lm_mov_reg = np.load(CASE_DIR / "landmarks_moving_eBH_reg1mm_zyx.npy")

with h5py.File(PTVREG_MAT, "r") as f:
    ptv_err = np.array(f["pt_errs_phys"]).reshape(-1)
    ptv_tre = float(np.array(f["TRE_phys"]).reshape(-1)[0])
    ptv_std = float(np.array(f["TREstd_case"]).reshape(-1)[0])

tail10 = ptv_err > 10.0
tail5 = ptv_err > 5.0

native_y_mid = int(np.clip(np.round(np.median(lm_fix_crop[:, 1])), 0, fixed_crop.shape[1] - 1))
native_y_tail = int(np.clip(np.round(np.median(lm_fix_crop[tail10, 1])), 0, fixed_crop.shape[1] - 1))
reg_y_mid = int(np.clip(np.round(np.median(lm_fix_reg[:, 1])), 0, fixed_reg.shape[1] - 1))
reg_y_tail = int(np.clip(np.round(np.median(lm_fix_reg[tail10, 1])), 0, fixed_reg.shape[1] - 1))

summary = {
    "fixed_crop_zyx": fixed_crop.shape,
    "moving_crop_zyx": moving_crop.shape,
    "fixed_reg1mm_zyx": fixed_reg.shape,
    "moving_reg1mm_zyx": moving_reg.shape,
    "ptvreg_mean_mm": ptv_tre,
    "ptvreg_std_mm": ptv_std,
    "ptvreg_median_mm": float(np.median(ptv_err)),
    "ptvreg_p95_mm": float(np.percentile(ptv_err, 95)),
    "ptvreg_max_mm": float(ptv_err.max()),
    "n_tail_gt5": int(tail5.sum()),
    "n_tail_gt10": int(tail10.sum()),
    "native_y_mid": native_y_mid,
    "native_y_tail": native_y_tail,
    "reg_y_mid": reg_y_mid,
    "reg_y_tail": reg_y_tail,
}
summary


In [ ]:
mask_cmap_fixed = ListedColormap([(0, 0, 0, 0), (0.0, 0.85, 0.35, 0.25)])
mask_cmap_moving = ListedColormap([(0, 0, 0, 0), (0.75, 0.25, 1.0, 0.25)])


def normalize_slice(slice_2d: np.ndarray) -> np.ndarray:
    arr = np.asarray(slice_2d, dtype=np.float32)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return np.zeros_like(arr)
    lo, hi = np.percentile(finite, [1, 99])
    if hi <= lo:
        lo, hi = float(finite.min()), float(finite.max())
    if hi <= lo:
        return np.zeros_like(arr)
    return np.clip((arr - lo) / (hi - lo), 0.0, 1.0)


def draw_orientation_labels(ax) -> None:
    box = dict(facecolor="black", alpha=0.35, pad=1, edgecolor="none")
    ax.text(0.50, 0.985, "Superior", ha="center", va="top", color="white", fontsize=7, transform=ax.transAxes, bbox=box)
    ax.text(0.50, 0.015, "Inferior", ha="center", va="bottom", color="white", fontsize=7, transform=ax.transAxes, bbox=box)
    ax.text(0.015, 0.50, "Patient R", ha="left", va="center", color="white", fontsize=7, rotation=90, transform=ax.transAxes, bbox=box)
    ax.text(0.985, 0.50, "Patient L", ha="right", va="center", color="white", fontsize=7, rotation=90, transform=ax.transAxes, bbox=box)


def draw_coronal_panel(ax, volume, mask, landmarks_zyx, y_idx, spacing_zyx, title, cmap, slab):
    y_idx = int(np.clip(y_idx, 0, volume.shape[1] - 1))
    img = np.flip(volume[:, y_idx, :], axis=0)
    msk = np.flip((mask[:, y_idx, :] > 0).astype(np.uint8), axis=0)
    aspect = float(spacing_zyx[0]) / float(spacing_zyx[2])
    ax.imshow(normalize_slice(img), cmap="gray", origin="lower", aspect=aspect, interpolation="nearest")
    ax.imshow(np.ma.masked_where(msk == 0, msk), cmap=cmap, origin="lower", aspect=aspect, interpolation="nearest", vmin=0, vmax=1)

    pts = np.asarray(landmarks_zyx, dtype=float)
    near_tail = (np.abs(pts[:, 1] - y_idx) <= slab) & tail10
    if near_tail.any():
        ax.scatter(pts[near_tail, 2], (volume.shape[0] - 1) - pts[near_tail, 0], s=28, facecolors="none", edgecolors="cyan", linewidths=1.2)

    near_all = np.abs(pts[:, 1] - y_idx) <= slab
    ax.set_title(f"{title}\ny={y_idx}; all LM slab n={int(near_all.sum())}; tail>10 n={int(near_tail.sum())}", fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])
    draw_orientation_labels(ax)


def draw_overlay(ax, fixed, moving, fixed_lm, moving_lm, y_idx, spacing_zyx, title, slab):
    y_idx = int(np.clip(y_idx, 0, fixed.shape[1] - 1))
    fsl = normalize_slice(np.flip(fixed[:, y_idx, :], axis=0))
    msl = normalize_slice(np.flip(moving[:, y_idx, :], axis=0))
    rgb = np.zeros((*fsl.shape, 3), dtype=float)
    rgb[..., 0] = msl
    rgb[..., 1] = fsl
    rgb[..., 2] = 0.15 * fsl
    aspect = float(spacing_zyx[0]) / float(spacing_zyx[2])
    ax.imshow(rgb, origin="lower", aspect=aspect, interpolation="nearest")

    for pts, color, marker in [(fixed_lm, "cyan", "o"), (moving_lm, "yellow", "x")]:
        pts = np.asarray(pts, dtype=float)
        near_tail = (np.abs(pts[:, 1] - y_idx) <= slab) & tail10
        if near_tail.any():
            ax.scatter(pts[near_tail, 2], (fixed.shape[0] - 1) - pts[near_tail, 0], s=32, c=color, marker=marker, linewidths=1.2)

    ax.set_title(f"{title}\ny={y_idx}; fixed=green moving=red", fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])
    draw_orientation_labels(ax)


In [ ]:
rows = [
    ("native crop median landmark plane", fixed_crop, moving_crop, fixed_mask_crop, moving_mask_crop, lm_fix_crop, lm_mov_crop, native_y_mid, (2.5, 0.645, 0.645), 4.0),
    ("native crop pTVReg >10 mm tail plane", fixed_crop, moving_crop, fixed_mask_crop, moving_mask_crop, lm_fix_crop, lm_mov_crop, native_y_tail, (2.5, 0.645, 0.645), 4.0),
    ("1 mm reg crop median landmark plane", fixed_reg, moving_reg, fixed_mask_reg, moving_mask_reg, lm_fix_reg, lm_mov_reg, reg_y_mid, (1.0, 1.0, 1.0), 5.0),
    ("1 mm reg crop pTVReg >10 mm tail plane", fixed_reg, moving_reg, fixed_mask_reg, moving_mask_reg, lm_fix_reg, lm_mov_reg, reg_y_tail, (1.0, 1.0, 1.0), 5.0),
]

fig, axes = plt.subplots(4, 3, figsize=(13, 15), constrained_layout=True)
fig.suptitle("COPD02 pTV-style cropped fixed/moving coronal audit — radiology orientation", fontsize=14)

for r, (label, fvol, mvol, fmask, mmask, flm, mlm, y, spacing, slab) in enumerate(rows):
    draw_coronal_panel(axes[r, 0], fvol, fmask, flm, y, spacing, label + " | fixed iBH", mask_cmap_fixed, slab)
    draw_coronal_panel(axes[r, 1], mvol, mmask, mlm, y, spacing, label + " | moving eBH", mask_cmap_moving, slab)
    draw_overlay(axes[r, 2], fvol, mvol, flm, mlm, y, spacing, label + " | overlay", slab)

legend = [
    Line2D([0], [0], marker="o", color="none", markeredgecolor="cyan", markerfacecolor="none", markersize=7, label="cyan circle: fixed iBH landmark with pTVReg TRE >10 mm"),
    Line2D([0], [0], marker="x", color="yellow", markersize=7, linestyle="none", label="yellow x: corresponding moving eBH landmark with pTVReg TRE >10 mm"),
    Line2D([0], [0], color="limegreen", lw=6, alpha=0.35, label="green mask overlay: fixed iBH lung mask"),
    Line2D([0], [0], color="orchid", lw=6, alpha=0.35, label="purple mask overlay: moving eBH lung mask"),
]
fig.legend(handles=legend, loc="lower center", ncol=2, fontsize=8, frameon=True)
fig.text(
    0.01,
    0.03,
    f"pTVReg best GPU case02: mean={ptv_tre:.3f} mm, median={np.median(ptv_err):.3f}, "
    f"p95={np.percentile(ptv_err,95):.3f}, max={ptv_err.max():.3f}, tail>10={int(tail10.sum())}/300. "
    "Display follows lungct_4dct_ptvprep_visual_audit.ipynb: flipped z, origin=lower, Superior top, Inferior bottom, Patient R left, Patient L right.",
    fontsize=8,
)
fig.savefig(FIG_PATH, dpi=180)
display(fig)
plt.close(fig)
display(Markdown('Saved figure: `$INRORSPLINE_REPO_ROOT/experiments/2026-08-03_dual_inr_copd_six_arm/preprocessing_audit/copd02_ptv_cropped_fixed_moving_coronal_audit.png`'))


## Readout

This visual audit is meant to catch gross preprocessing issues: wrong crop, phase swap, obvious truncation, or display-orientation confusion. It does not by itself prove or disprove subtle landmark annotation, interpolation, or local-correspondence issues.

For the current COPD02 pTVReg-best rerun, the outlier subset is small but severe: 13/300 landmarks have snap TRE >10 mm. In the displayed tail planes, the outlier landmarks remain inside the crop/lung region; the visible issue is a localized fixed-moving mismatch rather than an obvious crop truncation.